In [ ]:
#--- Rename SNP rsID to AoU within the SNP weight files
snps <- read.table(
  "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/AoU_UKB_SNP_Conversion_list_Autosomes.tab",
  header = TRUE,
  stringsAsFactors = FALSE
)

mdd <- read.table(
  "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/MDD_04.snpRes",
  header = TRUE,
  stringsAsFactors = FALSE
)

#-- #SNPs in the sbayesrc mdd snp file
library(tidyverse)
nrow(mdd) #SNPs in the sbayesrc mdd snp file
nrow(snps) # SNPs in the mapping file
mdd <- mdd %>% select(Name, A1, A1Effect)
snps <- snps %>% select(rsID_UKB_hg37, SNPID_AoU_hg38)
dat <- mdd %>% inner_join(snps, by = c("Name" = "rsID_UKB_hg37"))
dat <- dat %>% select(SNPID_AoU_hg38, A1, A1Effect)

library(data.table)
fwrite(dat, "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/MDD_04.txt",
       sep = "\t",   
       quote = FALSE,
       row.names = FALSE)  

In [ ]:
export REGION="us-central1"
export WORKSPACE_BUCKET="gs://fc-secure-cddd42d7-214c-4a32-9140-daa328e6beee"
export GENOTYPE_PREFIX="${WORKSPACE_BUCKET}/Genotype"
export IMAGE="${ARTIFACT_REGISTRY_DOCKER_REPO}/biocontainers/plink1.9:v1.90b6.6-181012-1-deb_cv1"
export MACHINE_TYPE="n1-standard-4"
export BOOT_DISK=50
export DISK_SIZE=200
export LOGGING_DIR="${WORKSPACE_BUCKET}/logs/"

for CHR in $(seq 1 22); do
  echo "Submitting job for chr${CHR} ..."

  aou_dsub \
    --regions "${REGION}" \
    --logging "${LOGGING_DIR}" \
    --name "Extr-${CHR}" \
    --image "${IMAGE}" \
    --machine-type "${MACHINE_TYPE}" \
    --boot-disk-size "${BOOT_DISK}" \
    --disk-size "${DISK_SIZE}" \
    --input BIM="${GENOTYPE_PREFIX}/UKBu_All_MAF01_hg38_chr${CHR}.bim" \
    --input FAM="${GENOTYPE_PREFIX}/UKBu_All_MAF01_hg38_chr${CHR}.fam" \
    --input BED="${GENOTYPE_PREFIX}/UKBu_All_MAF01_hg38_chr${CHR}.bed" \
    --input-recursive input_path="${WORKSPACE_BUCKET}/snp_weights/" \
    --output-recursive out_path="${WORKSPACE_BUCKET}/PGS/" \
    --env CHR="${CHR}" \
    --command "
for i in /mnt/data/input/gs/fc-secure-cddd42d7-214c-4a32-9140-daa328e6beee/snp_weights/Insomnia_01.txt; do
  score_name=\${i##*/}
  score_name=\${score_name%.txt}

  plink1.9 \
    --bed \"\${BED}\" \
    --bim \"\${BIM}\" \
    --fam \"\${FAM}\" \
    --score \"\$i\" 1 2 3 header sum \
    --out \"\${out_path}/\${score_name}_chr${CHR}\"
done
"
done

In [ ]:
export REGION="us-central1"
export WORKSPACE_BUCKET="gs://fc-secure-cddd42d7-214c-4a32-9140-daa328e6beee"
export GENOTYPE_PREFIX="${WORKSPACE_BUCKET}/Genotype"
export IMAGE="${ARTIFACT_REGISTRY_DOCKER_REPO}/biocontainers/plink1.9:v1.90b6.6-181012-1-deb_cv1"
export MACHINE_TYPE="n1-standard-4"
export BOOT_DISK=50
export DISK_SIZE=200
export LOGGING_DIR="${WORKSPACE_BUCKET}/logs/"

for CHR in $(seq 1 22); do
  echo "Submitting job for chr${CHR} ..."

  aou_dsub \
    --regions "${REGION}" \
    --logging "${LOGGING_DIR}" \
    --name "Extr-${CHR}" \
    --image "${IMAGE}" \
    --machine-type "${MACHINE_TYPE}" \
    --boot-disk-size "${BOOT_DISK}" \
    --disk-size "${DISK_SIZE}" \
    --input BIM="${GENOTYPE_PREFIX}/UKBu_All_MAF01_hg38_chr${CHR}.bim" \
    --input FAM="${GENOTYPE_PREFIX}/UKBu_All_MAF01_hg38_chr${CHR}.fam" \
    --input BED="${GENOTYPE_PREFIX}/UKBu_All_MAF01_hg38_chr${CHR}.bed" \
    --input-recursive input_path="${WORKSPACE_BUCKET}/snp_weights/" \
    --output-recursive out_path="${WORKSPACE_BUCKET}/PGS/" \
    --env CHR="${CHR}" \
    --command "
for i in /mnt/data/input/gs/fc-secure-cddd42d7-214c-4a32-9140-daa328e6beee/snp_weights/MDD_04.txt; do
  score_name=\${i##*/}
  score_name=\${score_name%.txt}

  plink1.9 \
    --bed \"\${BED}\" \
    --bim \"\${BIM}\" \
    --fam \"\${FAM}\" \
    --score \"\$i\"\
    --out \"\${out_path}/\${score_name}_v2_chr${CHR}\"
done
"
done

In [ ]:
library(data.table)

# ---------------------------------------------------------------------------
# Aggregate per-chromosome PLINK 1.9 PGS into genome-wide scores for BOTH:
#   - MDD_04          : scored WITHOUT `sum`  -> uses per-chr AVERAGE  (SCORE)
#   - MDD_04_nonavg   : scored WITH    `sum`  -> uses additive total   (SCORESUM)
# Then standardise each and correlate, to quantify whether the averaging
# convention materially changed individuals' rank ordering.
# ---------------------------------------------------------------------------

path    <- "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/PGS/"
out_dir <- "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/PGS/"

# label  : name for outputs
# stem   : filename stem before "_chr<N>.profile"
# valcol : which .profile column to aggregate
#            "SCORE"    = per-chromosome average  (the non-sum run)
#            "SCORESUM" = additive total          (the `sum` run)  [auto-detects SCORE1_SUM]
phenotypes <- list(
  list(label = "MDD_04_v2",        stem = "MDD_04_v2",        valcol = "SCORE"),
  list(label = "MDD_04", stem = "MDD_04", valcol = "SCORESUM")
)

process_phenotype <- function(label, stem, valcol) {

  files <- list.files(
    path,
    pattern    = paste0("^", stem, "_chr[0-9]+\\.profile$"),
    full.names = TRUE
  )
  if (length(files) == 0) {
    warning(sprintf("No files found for %s (stem '%s')", label, stem)); return(NULL)
  }
  if (length(files) != 22)
    warning(sprintf("%s: expected 22 chromosomes, found %d", label, length(files)))

  cat("Processing", label, "- found", length(files), "files (aggregating column:", valcol, ")\n")

  read_profile <- function(f) {
    chr <- as.integer(sub(".*_chr([0-9]+)\\.profile$", "\\1", basename(f)))
    d   <- fread(f)

    # resolve the requested value column (allow SCORESUM / SCORE1_SUM aliasing)
    col <- if (valcol == "SCORESUM") intersect(c("SCORESUM","SCORE1_SUM"), names(d))[1] else
           if (valcol %in% names(d)) valcol else NA_character_
    if (is.na(col))
      stop(sprintf("Column '%s' not found in %s (cols: %s)",
                   valcol, basename(f), paste(names(d), collapse = ", ")))

    cntcol    <- intersect(c("CNT","ALLELE_CT","NMISS_ALLELE_CT"), names(d))[1]
    snps_used <- if (!is.na(cntcol)) max(d[[cntcol]], na.rm = TRUE) / 2 else NA_real_

    list(
      scores = data.table(FID = d$FID, IID = d$IID, score = d[[col]], chr = chr),
      n_snps = data.table(chr = chr, snps_used = snps_used)
    )
  }

  parsed   <- lapply(files, read_profile)
  long     <- rbindlist(lapply(parsed, `[[`, "scores"))
  snpcount <- rbindlist(lapply(parsed, `[[`, "n_snps"))[order(chr)]

  cat("\nSNPs used per chromosome (", label, "):\n", sep = "")
  print(snpcount)
  cat(sprintf("TOTAL SNPs used across %d chromosomes: %s\n\n",
              nrow(snpcount), format(sum(snpcount$snps_used, na.rm = TRUE), big.mark = ",")))

  wide     <- dcast(long, FID + IID ~ chr, value.var = "score")
  chr_cols <- setdiff(names(wide), c("FID", "IID"))

  na_rows <- wide[!complete.cases(wide[, ..chr_cols])]
  if (nrow(na_rows) > 0)
    warning(sprintf("%s: %d individuals missing a score on >=1 chromosome - NOT zero-filling.",
                    label, nrow(na_rows)))

  pgs_col <- paste0(label, "_PGS")
  wide[, (pgs_col) := rowSums(.SD), .SDcols = chr_cols]   # NA propagates by design

  scores   <- wide[, c("FID", "IID", pgs_col), with = FALSE]
  out_file <- file.path(out_dir, paste0(label, "_scores.txt"))
  fwrite(scores, out_file, sep = "\t", quote = FALSE)
  cat("Wrote", out_file, "\n\n")

  list(scores = scores, snp_counts = snpcount, pgs_col = pgs_col)
}

# ---- run both ----
results <- lapply(phenotypes, function(p) process_phenotype(p$label, p$stem, p$valcol))
names(results) <- vapply(phenotypes, `[[`, character(1), "label")

# ---- compare the two scores: does the averaging convention change rank order? ----
# ---- compare the two scores: does the averaging convention change rank order? ----
r1 <- results[["MDD_04_v2"]]   # averaged (potential-bias) version
r2 <- results[["MDD_04"]]      # summed (correct) version

if (!is.null(r1) && !is.null(r2)) {
  m <- merge(r1$scores, r2$scores, by = c("FID", "IID"))
  v_old <- r1$pgs_col   # MDD_04_v2_PGS  (averaged)
  v_new <- r2$pgs_col   # MDD_04_PGS     (summed)

  m[, z_old := scale(get(v_old))[, 1]]
  m[, z_new := scale(get(v_new))[, 1]]

  cat("=========================================================\n")
  cat("Comparison: averaged (MDD_04_v2) vs summed (MDD_04)\n")
  cat("N individuals in both:", nrow(m), "\n")
  cat(sprintf("Pearson  (raw)         : %.5f\n", cor(m[[v_old]], m[[v_new]])))
  cat(sprintf("Pearson  (standardised): %.5f\n", cor(m$z_old, m$z_new)))
  cat(sprintf("Spearman (rank)        : %.5f\n", cor(m[[v_old]], m[[v_new]], method = "spearman")))
  cat("=========================================================\n")

  fwrite(m[, .(FID, IID, get(v_old), get(v_new), z_old, z_new)],
         file.path(out_dir, "MDD_04_score_comparison.txt"), sep = "\t", quote = FALSE)
}

In [ ]:
library(data.table)

# ---------------------------------------------------------------------------
# Aggregate per-chromosome PLINK 1.9 PGS (scored WITH `sum`) into genome-wide
# scores for MANY traits, then merge every trait into a single wide CSV.
# ---------------------------------------------------------------------------

path    <- "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/PGS/"
out_dir <- path

traits <- c(
  MDD      = "MDD_04",
  ADHD     = "ADHD_01",
  SCZ      = "SCZ_03",
  MIGRAINE = "Migraine_01",
  BIP      = "BIP_2025",
  INSOMNIA = "Insomnia_01",
  BMI      = "BMI_LOO"
)

# ---------------------------------------------------------------------------
aggregate_pgs <- function(label, stem) {

  files <- list.files(
    path,
    pattern    = paste0("^", stem, "_chr[0-9]+\\.profile$"),
    full.names = TRUE
  )
  if (length(files) == 0) {
    warning(sprintf("[%s] no files found for stem '%s' - skipping", label, stem))
    return(NULL)
  }
  if (length(files) != 22)
    warning(sprintf("[%s] expected 22 chromosomes, found %d", label, length(files)))

  read_profile <- function(f) {
    chr <- as.integer(sub(".*_chr([0-9]+)\\.profile$", "\\1", basename(f)))
    d   <- fread(f)

    # summed convention only (allow SCORESUM / SCORE1_SUM aliasing)
    col <- intersect(c("SCORESUM", "SCORE1_SUM"), names(d))[1]
    if (is.na(col))
      stop(sprintf("[%s] no SCORESUM/SCORE1_SUM in %s (cols: %s). Was it scored with `sum`?",
                   label, basename(f), paste(names(d), collapse = ", ")))

    cntcol    <- intersect(c("CNT", "ALLELE_CT", "NMISS_ALLELE_CT"), names(d))[1]
    snps_used <- if (!is.na(cntcol)) max(d[[cntcol]], na.rm = TRUE) / 2 else NA_real_

    list(
      scores = data.table(FID = d$FID, IID = d$IID, score = d[[col]], chr = chr),
      n_snps = data.table(chr = chr, snps_used = snps_used)
    )
  }

  parsed   <- lapply(files, read_profile)
  long     <- rbindlist(lapply(parsed, `[[`, "scores"))
  snpcount <- rbindlist(lapply(parsed, `[[`, "n_snps"))[order(chr)]

  wide     <- dcast(long, FID + IID ~ chr, value.var = "score")
  chr_cols <- setdiff(names(wide), c("FID", "IID"))

  na_rows <- wide[!complete.cases(wide[, ..chr_cols])]
  if (nrow(na_rows) > 0)
    warning(sprintf("[%s] %d individuals missing a score on >=1 chromosome - NOT zero-filling.",
                    label, nrow(na_rows)))

  pgs_col <- paste0(label, "_PGS")
  wide[, (pgs_col) := rowSums(.SD), .SDcols = chr_cols]   # NA propagates by design

  cat(sprintf("%-9s | %2d chr | %14s SNPs | N = %d\n",
              label, nrow(snpcount),
              format(sum(snpcount$snps_used, na.rm = TRUE), big.mark = ","),
              nrow(wide)))

  wide[, c("FID", "IID", pgs_col), with = FALSE]
}

# ---- run every trait ----
cat("Aggregating per-trait genome-wide PGS:\n")
score_list <- Map(aggregate_pgs, traits, traits)
score_list <- Filter(Negate(is.null), score_list)

# ---- merge into one wide table (full outer join keeps all individuals) ----
combined <- Reduce(function(x, y) merge(x, y, by = c("FID", "IID"), all = TRUE), score_list)

pgs_cols <- setdiff(names(combined), c("FID", "IID"))
n_any_na <- sum(!complete.cases(combined[, ..pgs_cols]))
if (n_any_na > 0)
  cat(sprintf("\nNote: %d individuals are missing a PGS for >=1 trait (kept as NA).\n", n_any_na))

# ---- write combined CSV ----
out_file <- file.path(out_dir, "all_traits_PGS.csv")
fwrite(combined, out_file)
cat(sprintf("\nWrote %d individuals x %d traits to %s\n",
            nrow(combined), length(pgs_cols), out_file))